# v7 Python Authoring Quickstart

This notebook shows the new Python authoring layer on top of the existing `v7` training pipeline.

How it works:
- Python owns the project spec: model dimensions, template choice, data source, and train config.
- `TrainingProject.materialize()` calls the existing `version/v7/scripts/ck_run_v7.py init ... --generate-ir --generate-runtime` path.
- `TrainingProject.train()` calls the existing `version/v7/scripts/ck_run_v7.py train ...` path.
- The actual runtime remains generated C code plus `libtrain.so`; Python is the authoring and orchestration layer.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ckernel_engine").exists() and (REPO_ROOT.parent / "ckernel_engine").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "ckernel_engine").exists():
    raise RuntimeError(f"Could not find repo root from cwd={Path.cwd()}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT


In [ ]:
from ckernel_engine.v7 import (
    DataSource,
    MaterializeOptions,
    TemplateSpec,
    TinyModelSpec,
    TokenizerPlan,
    TrainConfig,
    TrainingProject,
)


In [ ]:
project = TrainingProject(
    run_name="python-ui-notebook-demo",
    model=TinyModelSpec(
        init="xavier_uniform",
        layers=2,
        vocab_size=256,
        embed_dim=128,
        hidden_dim=256,
        num_heads=8,
        num_kv_heads=4,
        context_len=128,
    ),
    template=TemplateSpec.builtin_template("qwen3"),
    tokenizer=TokenizerPlan(
        family="runtime_default",
        notes="Notebook keeps tokenizer ownership in the existing v7 runtime path.",
    ),
)

project


In [ ]:
materialize_result = project.materialize(
    MaterializeOptions(
        generate_ir=True,
        generate_runtime=True,
        strict=True,
    )
)

materialize_result


In [ ]:
train_result = project.train(
    DataSource.inline_text(
        "C-Kernel-Engine notebook quickstart.",
        description="small inline training text",
    ),
    TrainConfig(
        backend="ck",
        strict=True,
        epochs=1,
        seq_len=8,
        total_tokens=64,
        grad_accum=2,
        lr=5e-4,
        parity_regimen="suggest",
    ),
)

train_result


In [ ]:
run_dir = train_result.run_dir
sorted(p.name for p in run_dir.iterdir())


## What To Inspect

After the cells above run, the important artifacts are:
- `python_authoring_plan.json`: Python-side project spec and command history
- `weights_manifest.json`: manifest-first training state and embedded template contract
- `ir1_train_forward.json`: train forward IR
- `ir2_train_backward.json`: backward IR
- `layout_train.json`: planned memory layout
- `generated_train_runtime_v7.c`: generated training runtime source
- `libtrain.so`: compiled generated runtime
- `train_e2e_latest.json`: latest train report exported by the existing `v7` runner
